In [17]:
import setup

setup.init_django()

In [18]:
from decouple import config

In [19]:
from analytics.models import PageView
from blog.models import BlogPost
from rag import db as rag_db, settings as rag_settings

In [20]:
from sqlalchemy import (
    create_engine,
    inspect,
)

from llama_index.core import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.core.retrievers import NLSQLRetriever

In [21]:
# initialize default LlamaIndex settings
rag_settings.init()
# get pooled Neon database string from .env or env vars
vector_database_url = rag_db.get_database_url(use_pooling=True)

In [22]:
engine = create_engine(vector_database_url)
engine = create_engine(
    vector_database_url,
    pool_pre_ping=True,
    pool_recycle=1800,   # Recycle every 30 mins
)

In [23]:
inspect(engine).get_table_names()

['django_migrations',
 'auth_user_groups',
 'products_product',
 'auth_permission',
 'django_content_type',
 'auth_user',
 'django_admin_log',
 'blog_blogpost',
 'auth_group_permissions',
 'auth_user_user_permissions',
 'django_session',
 'auth_group',
 'products_embedding',
 'analytics_pageview']

In [24]:
tables = []
models = [BlogPost, PageView]
for model in models:
    table = model._meta.db_table
    tables.append(table)

In [25]:
tables

['blog_blogpost', 'analytics_pageview']

In [26]:
sql_database = SQLDatabase(engine, include_tables=tables)

In [27]:
sql_query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,
    tables=tables,
)

In [28]:
response = sql_query_engine.query("What blog post has the most views?")
print(str(response))

The blog post with the most views is "Blog Post 2," which has been viewed 2,366 times.


In [29]:
for node in response.source_nodes:
    print(node.node.get_content())

[('Blog Post 2', 2366)]


In [30]:
nl_sql_retriever = NLSQLRetriever(
    sql_database, tables=tables, return_raw=True
)

r = nl_sql_retriever.retrieve("What is my least most viewed blog post?")

In [31]:
print(r)
for node in r:
    print(node)
    print(node.metadata)

[NodeWithScore(node=TextNode(id_='76ea6cb0-a12a-402e-9c10-5a082074bef0', embedding=None, metadata={'sql_query': 'SELECT blog_blogpost.title, COUNT(analytics_pageview.id) AS view_count FROM blog_blogpost LEFT JOIN analytics_pageview ON blog_blogpost.id = analytics_pageview.post_id GROUP BY blog_blogpost.id ORDER BY view_count ASC LIMIT 1;', 'result': [('New Blog', 0)], 'col_keys': ['title', 'view_count']}, excluded_embed_metadata_keys=['sql_query', 'result', 'col_keys'], excluded_llm_metadata_keys=['sql_query', 'result', 'col_keys'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text="[('New Blog', 0)]", mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=None)]
Node ID: 76ea6cb0-a12a-402e-9c10-5a082074bef0
Text: [('New Blog', 0)]
Score: None

{'sql_query': 'SELECT blog_blogpost.title, COUNT(analytics_pageview.id) AS view_count FROM blog_blogpost LEFT JOIN analytics_p